# Phase 3 — Gemma Attention Extraction (Low-Memory)

Memory-safe Phase-3 notebook for **`google/gemma-2-2b-it`**.

This preserves the final Qwen extraction design:

- 30 query tokens immediately before the pivot
- up to 250 pre-pivot key tokens
- shorter contexts are left-padded with zeros
- longer contexts retain the 250 most recent pre-pivot keys
- extract attention patterns and per-head `z`
- only `correction` / `hallucination_pivot` records with at least 30 pre-pivot tokens

Expected Gemma sample: **76 usable records = 13 corrections + 63 hallucination pivots**.

The implementation is changed only for memory safety: activations are streamed through hooks, sliced immediately, moved to CPU, and saved one record at a time instead of using `run_with_cache()` or retaining the full dataset in RAM.

## 1. Install, authenticate, and verify GPU

In [2]:
!pip install -q "transformer_lens>=3.1,<4" transformers huggingface_hub

import os, gc, json, time, shutil, platform
from pathlib import Path
from collections import Counter
from importlib.metadata import version

import numpy as np
import torch
import transformer_lens
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing."

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN.")

os.environ["HF_TOKEN"] = HF_TOKEN

print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("PyTorch:", torch.__version__)
print("TransformerLens:", version("transformer-lens"))
print("Python:", platform.python_version())
print("HF token loaded: yes")

GPU: Tesla T4
VRAM: 15.6 GB
PyTorch: 2.10.0+cu128
TransformerLens: 3.7.1
Python: 3.12.13
HF token loaded: yes


## 2. Locate the final Gemma Phase-2 dataset

In [3]:
# ============================================================
# PATHS
# ============================================================

from pathlib import Path

# Final reviewed Gemma Phase-2 file
PHASE2_PATH = (
    "/kaggle/input/notebooks/gunjalgupta/phase-2-gemma/"
    "pivot_research_gemma/phase2_labeled_gemma.jsonl"
)

# Verify input exists
if not Path(PHASE2_PATH).exists():
    raise FileNotFoundError(
        f"Phase-2 file not found:\n{PHASE2_PATH}"
    )

# Phase-3 outputs
OUTPUT_DIR = Path("/kaggle/working/pivot_research_gemma")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = OUTPUT_DIR / "phase3_gemma_records"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = OUTPUT_DIR / "phase3_metadata_gemma.jsonl"
ERRORS_PATH = OUTPUT_DIR / "phase3_errors_gemma.jsonl"
MANIFEST_PATH = OUTPUT_DIR / "phase3_manifest_gemma.json"

print("Phase-2 input:")
print(PHASE2_PATH)

print("\nPhase-3 output directory:")
print(OUTPUT_DIR)

print("\nRecord tensors:")
print(TEMP_DIR)

print("\nInput file verified successfully.")

Phase-2 input:
/kaggle/input/notebooks/gunjalgupta/phase-2-gemma/pivot_research_gemma/phase2_labeled_gemma.jsonl

Phase-3 output directory:
/kaggle/working/pivot_research_gemma

Record tensors:
/kaggle/working/pivot_research_gemma/phase3_gemma_records

Input file verified successfully.


## 3. Freeze the usable sample

In [4]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

all_records = load_jsonl(PHASE2_PATH)

QUERY_WINDOW_SIZE = 30
MAX_KEY_CONTEXT = 250

usable_records = [
    r for r in all_records
    if r.get("label") in {"correction", "hallucination_pivot"}
    and int(r.get("pivot_token_start", -1)) >= QUERY_WINDOW_SIZE
]

counts = Counter(r["label"] for r in usable_records)

print("Total Phase-2 records:", len(all_records))
print("Usable:", len(usable_records))
print("Labels:", dict(counts))

EXPECTED_USABLE = 76
EXPECTED_CORRECTIONS = 13
EXPECTED_HALLUCINATIONS = 63

assert len(usable_records) == EXPECTED_USABLE, (
    f"Expected {EXPECTED_USABLE} usable records; found {len(usable_records)}."
)
assert counts["correction"] == EXPECTED_CORRECTIONS
assert counts["hallucination_pivot"] == EXPECTED_HALLUCINATIONS

def category_of(r):
    if r.get("category"):
        return r["category"]
    pid = r["prompt_id"]
    if pid.startswith("math_"):
        return "math"
    if pid.startswith("logic_"):
        return "logic"
    return "misdirection"

print("\nCorrections by category:")
for cat in ("misdirection", "math", "logic"):
    n = sum(1 for r in usable_records
            if r["label"] == "correction" and category_of(r) == cat)
    print(f"  {cat:<14} {n}")

lens = sorted(int(r["pivot_token_start"]) for r in usable_records)
print("\nPre-pivot tokens:")
print("  min:", lens[0])
print("  median:", lens[len(lens)//2])
print("  max:", lens[-1])
print("  truncated (>250):", sum(x > MAX_KEY_CONTEXT for x in lens))
print("  padded (<250):", sum(x < MAX_KEY_CONTEXT for x in lens))

Total Phase-2 records: 108
Usable: 76
Labels: {'hallucination_pivot': 63, 'correction': 13}

Corrections by category:
  misdirection   9
  math           3
  logic          1

Pre-pivot tokens:
  min: 34
  median: 97
  max: 229
  truncated (>250): 0
  padded (<250): 76


## 4. Load Gemma with TransformerLens

This intentionally keeps the same `HookedTransformer` interface used by the Qwen extraction for cross-model comparability.

In [5]:
# ============================================================
# 4. LOAD GEMMA — LOW MEMORY / NO WEIGHT PROCESSING
# ============================================================

import gc
import torch
from transformer_lens import HookedTransformer

MODEL_NAME = "google/gemma-2-2b-it"

# Gemma was trained primarily in bfloat16.
MODEL_DTYPE = torch.bfloat16

# Clean anything left in memory before loading
gc.collect()
torch.cuda.empty_cache()

print("Loading:", MODEL_NAME)
print("dtype:", MODEL_DTYPE)
print("Using from_pretrained_no_processing()")

# IMPORTANT:
# HF_TOKEN is already available through:
# os.environ["HF_TOKEN"] = HF_TOKEN
#
# Do NOT pass token=HF_TOKEN here.

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_NAME,
    device="cuda",
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)

model.eval()

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# Model dimensions
# ------------------------------------------------------------

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads
D_HEAD = model.cfg.d_head
D_MODEL = model.cfg.d_model

print("\nModel loaded successfully!")
print("=" * 60)
print("Layers:", N_LAYERS)
print("Heads:", N_HEADS)
print("d_head:", D_HEAD)
print("d_model:", D_MODEL)
print("dtype:", MODEL_DTYPE)

print("\nGPU memory after load:")
print(
    f"Allocated: "
    f"{torch.cuda.memory_allocated()/1e9:.2f} GB"
)
print(
    f"Reserved:  "
    f"{torch.cuda.memory_reserved()/1e9:.2f} GB"
)
print(
    f"Total:     "
    f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB"
)

Loading: google/gemma-2-2b-it
dtype: torch.bfloat16
Using from_pretrained_no_processing()


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer

Model loaded successfully!
Layers: 26
Heads: 8
d_head: 256
d_model: 2304
dtype: torch.bfloat16

GPU memory after load:
Allocated: 6.47 GB
Reserved:  6.67 GB
Total:     15.64 GB


## 5. Verify Phase-2 token indices under the Gemma tokenizer

In [6]:
warnings = []

for r in usable_records:
    ids = model.tokenizer.encode(r["generated_text"], add_special_tokens=False)
    p = int(r["pivot_token_start"])

    if p < QUERY_WINDOW_SIZE or p >= len(ids):
        warnings.append({
            "key": f'{r["prompt_id"]}#{r.get("sample_idx")}',
            "pivot": p,
            "seq_len": len(ids),
        })

print("Alignment warnings:", len(warnings))

if warnings:
    print(warnings[:10])
    raise RuntimeError(
        "Token alignment failed. Phase 2 and Phase 3 must use the same Gemma tokenizer."
    )

print("\nSample pivot neighborhoods:")
for r in usable_records[:5]:
    ids = model.tokenizer.encode(r["generated_text"], add_special_tokens=False)
    p = int(r["pivot_token_start"])
    text = model.tokenizer.decode(ids[max(0,p-6):min(len(ids),p+8)])
    print(f'{r["prompt_id"]}#{r.get("sample_idx")} [{r["label"]}]')
    print(" ", repr(text))

Alignment warnings: 0

Sample pivot neighborhoods:
mis_04#4 [hallucination_pivot]
  ' to double check. \n\nWait, what if I take all the'
mis_04#5 [hallucination_pivot]
  ' of a solution. \n\nWait, 20 minutes is not'
mis_06#1 [correction]
  " since I took them.  Wait, that's a reasonable doubt"
mis_07#14 [correction]
  '3 cows remaining. \n\nWait, isn\'t the statement "'
mis_08#2 [hallucination_pivot]
  ' some warm heat. \n\n\nHmm, the stove will give heat,'


## 6. Low-memory hook-based extraction

This version does **not** call `run_with_cache()`. Each hook slices only the needed activation window and moves it to CPU immediately.

In [7]:
def extract_attention_features(record):
    input_ids = model.tokenizer.encode(
        record["generated_text"],
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    seq_len = int(input_ids.shape[1])
    pivot_idx = int(record["pivot_token_start"])

    if pivot_idx < QUERY_WINDOW_SIZE:
        raise ValueError(f"pivot_idx={pivot_idx} is too early")
    if pivot_idx >= seq_len:
        raise ValueError(f"pivot_idx={pivot_idx} >= seq_len={seq_len}")

    query_start = pivot_idx - QUERY_WINDOW_SIZE
    query_end = pivot_idx
    key_real_start = max(0, pivot_idx - MAX_KEY_CONTEXT)
    key_end = pivot_idx

    n_real_keys = key_end - key_real_start
    valid_key_start = MAX_KEY_CONTEXT - n_real_keys
    n_dropped_early_keys = max(0, pivot_idx - MAX_KEY_CONTEXT)

    weights_by_layer = [None] * N_LAYERS
    outputs_by_layer = [None] * N_LAYERS

    def make_pattern_hook(layer_idx):
        def hook_fn(pattern, hook):
            sliced = pattern[0, :, query_start:query_end, key_real_start:key_end]
            sliced_cpu = sliced.detach().to("cpu", dtype=torch.float32).numpy()
            padded = np.zeros(
                (N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT),
                dtype=np.float32,
            )
            padded[:, :, valid_key_start:] = sliced_cpu
            weights_by_layer[layer_idx] = padded
            return None
        return hook_fn

    def make_z_hook(layer_idx):
        def hook_fn(z, hook):
            sliced = z[0, query_start:query_end, :, :]
            outputs_by_layer[layer_idx] = sliced.detach().to("cpu", dtype=torch.float32).numpy()
            return None
        return hook_fn

    hooks = []
    for layer in range(N_LAYERS):
        hooks.append((f"blocks.{layer}.attn.hook_pattern", make_pattern_hook(layer)))
        hooks.append((f"blocks.{layer}.attn.hook_z", make_z_hook(layer)))

    with torch.inference_mode():
        model.run_with_hooks(input_ids, fwd_hooks=hooks, return_type=None)

    missing_w = [i for i, x in enumerate(weights_by_layer) if x is None]
    missing_z = [i for i, x in enumerate(outputs_by_layer) if x is None]
    if missing_w:
        raise RuntimeError(f"Pattern hooks missing layers: {missing_w}")
    if missing_z:
        raise RuntimeError(f"Z hooks missing layers: {missing_z}")

    attn_weights = np.stack(weights_by_layer, axis=0)
    attn_outputs = np.stack(outputs_by_layer, axis=0)

    del input_ids, weights_by_layer, outputs_by_layer, hooks
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "attn_weights": attn_weights,
        "attn_outputs": attn_outputs,
        "pivot_idx": pivot_idx,
        "seq_len": seq_len,
        "valid_key_start": valid_key_start,
        "n_real_keys": n_real_keys,
        "n_dropped_early_keys": n_dropped_early_keys,
    }

## 7. Dry run on one record — do not skip

In [8]:
test_record = usable_records[0]
print(f'Testing {test_record["prompt_id"]}#{test_record.get("sample_idx")} [{test_record["label"]}]')

before_alloc = torch.cuda.memory_allocated()
before_reserved = torch.cuda.memory_reserved()

t0 = time.time()
test = extract_attention_features(test_record)
elapsed = time.time() - t0

after_alloc = torch.cuda.memory_allocated()
after_reserved = torch.cuda.memory_reserved()

assert test["attn_weights"].shape == (N_LAYERS, N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT)
assert test["attn_outputs"].shape == (N_LAYERS, QUERY_WINDOW_SIZE, N_HEADS, D_HEAD)

print(f"Elapsed: {elapsed:.2f}s")
print("weights:", test["attn_weights"].shape)
print("outputs:", test["attn_outputs"].shape)
print("real keys:", test["n_real_keys"])
print("dropped early keys:", test["n_dropped_early_keys"])
print("Allocated before/after:", f"{before_alloc/1e9:.2f} / {after_alloc/1e9:.2f} GB")
print("Reserved before/after:", f"{before_reserved/1e9:.2f} / {after_reserved/1e9:.2f} GB")

mb = (test["attn_weights"].nbytes + test["attn_outputs"].nbytes) / 1e6
print(f"Stored CPU data per record: {mb:.2f} MB")
print(f"Estimated 76-record time: {elapsed*76/60:.1f} min")

del test
gc.collect()
torch.cuda.empty_cache()
print("Dry run passed.")

Testing mis_04#4 [hallucination_pivot]
Elapsed: 1.33s
weights: (26, 8, 30, 250)
outputs: (26, 30, 8, 256)
real keys: 99
dropped early keys: 0
Allocated before/after: 6.47 / 6.48 GB
Reserved before/after: 6.67 / 6.70 GB
Stored CPU data per record: 12.63 MB
Estimated 76-record time: 1.7 min
Dry run passed.


## 8. Extract all records and save each one immediately

Each successful record is written to its own compressed NPZ file so RAM usage stays approximately constant.

In [9]:
RESET_TEMP_OUTPUTS = True

if RESET_TEMP_OUTPUTS and TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)

metadata = []
errors = []
start_time = time.time()

for source_idx, record in enumerate(tqdm(usable_records, desc="Extracting Gemma attention")):
    try:
        result = extract_attention_features(record)
        out_idx = len(metadata)
        record_path = TEMP_DIR / f"record_{out_idx:03d}.npz"

        np.savez_compressed(
            record_path,
            weights=result["attn_weights"],
            outputs=result["attn_outputs"],
        )

        metadata.append({
            "idx": out_idx,
            "source_idx": source_idx,
            "record_file": record_path.name,
            "prompt_id": record["prompt_id"],
            "category": category_of(record),
            "sample_idx": record.get("sample_idx"),
            "label": record["label"],
            "label_method": record.get("label_method"),
            "human_verified": record.get("human_verified", False),
            "pivot_type": record.get("pivot_type"),
            "pivot_phrase": record.get("pivot_phrase"),
            "pivot_token_start": int(record["pivot_token_start"]),
            "query_window_size": QUERY_WINDOW_SIZE,
            "max_key_context": MAX_KEY_CONTEXT,
            "valid_key_start": result["valid_key_start"],
            "n_real_keys": result["n_real_keys"],
            "n_dropped_early_keys": result["n_dropped_early_keys"],
            "seq_len": result["seq_len"],
            "model_name": MODEL_NAME,
        })

        del result
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        key = f'{record.get("prompt_id")}#{record.get("sample_idx")}'
        print(f"\nERROR {key}: {type(e).__name__}: {e}")
        errors.append({
            "source_idx": source_idx,
            "prompt_id": record.get("prompt_id"),
            "sample_idx": record.get("sample_idx"),
            "label": record.get("label"),
            "error_type": type(e).__name__,
            "error": str(e),
        })

elapsed = time.time() - start_time

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    for m in metadata:
        f.write(json.dumps(m, ensure_ascii=False) + "\n")

if errors:
    with open(ERRORS_PATH, "w", encoding="utf-8") as f:
        for e in errors:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

print("\nExtraction finished.")
print(f"Elapsed: {elapsed/60:.1f} min")
print("Successful:", len(metadata))
print("Errors:", len(errors))
print("Metadata:", METADATA_PATH)

Extracting Gemma attention:   0%|          | 0/76 [00:00<?, ?it/s]


Extraction finished.
Elapsed: 3.1 min
Successful: 76
Errors: 0
Metadata: /kaggle/working/pivot_research_gemma/phase3_metadata_gemma.jsonl


## 9. Save reproducibility manifest

In [10]:
manifest = {
    "model_name": MODEL_NAME,
    "dtype": str(MODEL_DTYPE),
    "transformer_lens_version": version("transformer-lens"),
    "torch_version": torch.__version__,
    "query_window_size": QUERY_WINDOW_SIZE,
    "max_key_context": MAX_KEY_CONTEXT,
    "n_layers": N_LAYERS,
    "n_heads": N_HEADS,
    "d_head": D_HEAD,
    "d_model": D_MODEL,
    "phase2_path": str(PHASE2_PATH),
    "n_phase2_records": len(all_records),
    "n_usable_requested": len(usable_records),
    "n_extracted": len(metadata),
    "n_errors": len(errors),
    "label_counts": dict(Counter(m["label"] for m in metadata)),
    "category_counts": dict(Counter(m["category"] for m in metadata)),
    "elapsed_minutes": elapsed / 60,
    "storage_format": "one compressed npz per record",
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Manifest:", MANIFEST_PATH)
print("Labels:", dict(Counter(m["label"] for m in metadata)))

Manifest: /kaggle/working/pivot_research_gemma/phase3_manifest_gemma.json
Labels: {'hallucination_pivot': 63, 'correction': 13}


## 10. Final sanity checks

This validates the record-wise files one at a time and never loads the full tensor dataset into memory.

In [11]:
assert metadata, "No successful records found."

padding_failures = []
shape_failures = []
full_context_rows = []
truncated_rows = []

for m in tqdm(metadata, desc="Validating saved records"):
    path = TEMP_DIR / m["record_file"]
    with np.load(path) as d:
        w = d["weights"]
        z = d["outputs"]

        ew = (N_LAYERS, N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT)
        ez = (N_LAYERS, QUERY_WINDOW_SIZE, N_HEADS, D_HEAD)

        if w.shape != ew or z.shape != ez:
            shape_failures.append((m["idx"], w.shape, z.shape))
            continue

        vks = int(m["valid_key_start"])
        if vks > 0:
            pad_sum = float(np.abs(w[:, :, :, :vks]).sum())
            if pad_sum != 0.0:
                padding_failures.append((m["idx"], pad_sum))

        row_sum = float(w[0, 0, QUERY_WINDOW_SIZE//2, vks:].sum())
        if m["n_dropped_early_keys"] == 0:
            full_context_rows.append(row_sum)
        else:
            truncated_rows.append(row_sum)

print("Shape failures:", len(shape_failures))
print("Padding failures:", len(padding_failures))
assert not shape_failures, shape_failures[:5]
assert not padding_failures, padding_failures[:5]

if full_context_rows:
    print("Full-context row sums:",
          f"min={min(full_context_rows):.4f}",
          f"median={np.median(full_context_rows):.4f}",
          f"max={max(full_context_rows):.4f}")
if truncated_rows:
    print("Truncated row sums:",
          f"min={min(truncated_rows):.4f}",
          f"median={np.median(truncated_rows):.4f}",
          f"max={max(truncated_rows):.4f}")

final_counts = Counter(m["label"] for m in metadata)
print("\nFinal extracted counts:", dict(final_counts))

assert len(metadata) == 76
assert final_counts["correction"] == 13
assert final_counts["hallucination_pivot"] == 63

print("\nREADY FOR PHASE 4")
print("Record tensors:", TEMP_DIR)
print("Metadata:", METADATA_PATH)
print("Manifest:", MANIFEST_PATH)

Validating saved records:   0%|          | 0/76 [00:00<?, ?it/s]

Shape failures: 0
Padding failures: 0
Full-context row sums: min=0.9979 median=0.9998 max=1.0021

Final extracted counts: {'hallucination_pivot': 63, 'correction': 13}

READY FOR PHASE 4
Record tensors: /kaggle/working/pivot_research_gemma/phase3_gemma_records
Metadata: /kaggle/working/pivot_research_gemma/phase3_metadata_gemma.jsonl
Manifest: /kaggle/working/pivot_research_gemma/phase3_manifest_gemma.json


## Phase-4 handoff

Use these outputs together:

- `phase3_gemma_records/record_000.npz` ... `record_075.npz`
- `phase3_metadata_gemma.jsonl`
- `phase3_manifest_gemma.json`

Each record file contains:

- `weights`: `[layers, heads, 30, 250]`
- `outputs`: `[layers, 30, heads, d_head]`

The record-wise format is intentional to keep Kaggle RAM usage low. Phase 4 should iterate over the files rather than loading all raw tensors into memory at once.